# ALACC Truck Special Generators comparison

This notebook compares the trucks special generation trips with those of the SW model 

In [106]:
import pandas as pd 
import numpy as np 

import geopandas as gpd
import openmatrix as omx

# Bi-county Special Generation Data

In [157]:
# In Alacc scenario path: nonres/Inputs/Calib/PORT_SG_2015.DBF"
# File was converted to CSV using CUBE and save to BOX 
path = "../data/external/ccta/PORT_SG_2015.csv"
df = pd.read_csv(path, header = None).dropna()
df.columns = ["taz", "SMALL", "MEDIUM", "COMBO"]
df["taz"] = df["taz"].astype(float).astype(int)
df = df.set_index("taz")

In [159]:
df["total"] = df.sum(axis = 1)
BCM = add_total_row(df[df["total"] > 0])
print("Table 1. Bi-county Model \nPort of Oakland Special Generator")
BCM.style.format("{:,.0f}")

Table 1. Bi-county Model 
Port of Oakland Special Generator


,SMALL,MEDIUM,COMBO,total
2833,74,194,953,"2,442"
2957,126,324,"1,585","4,070"
2967,50,124,612,"1,572"
3163,100,254,"1,245","3,198"
3165,70,186,901,"2,314"
3166,40,104,508,"1,304"
3167,16,40,198,508
TOTAL,476,"1,226","6,002","15,408"


Note: All these TAZs are around the Port of Oakland.

# State Wide Model 

In [174]:
# SW Trip Generation projected to TM-1.6 Zoning system
path = "../data/processed/truck_trip_generation_zone.csv"
sw_generation = pd.read_csv(path)
sw_generation =sw_generation.set_index("TAZ1454")

In [175]:
# These TAZs are comparable with the the area covered by the SPECIAL GENERATORS in the Bi-county Model
# This inspection was done manually. 
port_of_oakland_tazs = [988,965, 966]

sw_generation["small"] = sw_generation[sw_generation.columns[sw_generation.columns.str.contains(r"^LT.*production$")]].sum(axis = 1)
sw_generation["medium"] = sw_generation[sw_generation.columns[sw_generation.columns.str.contains(r"^MT.*production$")]].sum(axis = 1)
sw_generation["large"] = sw_generation[sw_generation.columns[sw_generation.columns.str.contains(r"^HT.*production$")]].sum(axis = 1)
sw_generation["total"] = sw_generation[["small", "medium", "large"]].sum(axis =1)

production_port_of_oakland = sw_generation[sw_generation.index.isin(port_of_oakland_tazs)][["small", "medium", "large"]]

In [176]:
port_of_oakland_tazs = [988,965, 966]
production_port_of_oakland = sw_generation[sw_generation.index.isin(port_of_oakland_tazs)][["small", "medium", "large"]]

In [177]:
#Statewide Transportation Logistic Nodes (TLN) projected to TM-1.6 Zoning System
path = "../data/processed/truck_trip_generation_tln.csv"
tln = pd.read_csv(path)

tln["small"] = tln[tln.columns[tln.columns.str.contains(r"^LT.*production$")]].sum(axis = 1)
tln["medium"] = tln[tln.columns[tln.columns.str.contains(r"^MT.*production$")]].sum(axis = 1)
tln["large"] = tln[tln.columns[tln.columns.str.contains(r"^HT.*production$")]].sum(axis = 1)
tln["total"] = tln[["small", "medium", "large"]].sum(axis =1)

In [178]:
tln[["TAZ1454", "small", "medium", "large"]]

,TAZ1454,small,medium,large
0,142.0,0.0,42.229202,129.498908
1,313.0,0.0,14.307400,56.660299
2,965.0,0.0,510.502294,1015.344990
3,1062.0,0.0,135.852806,344.753600
4,239.0,0.0,66.932599,173.411304
5,874.0,0.0,4.811900,29.225001


In [179]:
sw_port_of_oakland = production_port_of_oakland.merge(
    tln[["TAZ1454", "small", "medium", "large"]], 
    left_index = True, 
    right_on = "TAZ1454",
    how = 'left', 
    suffixes=("_taz", "_tln")
).set_index("TAZ1454").fillna(0)

sw_port_of_oakland["total"] = sw_port_of_oakland.sum(axis =1)
sw_port_of_oakland.index = sw_port_of_oakland.index.astype(int)
sw_port_of_oakland = add_total_row(sw_port_of_oakland)
print("Table 2. SW Model \nPort of Oakland TAZ and TLN demand")
sw_port_of_oakland.style.format("{:,.0f}")

Table 2. SW Model 
Port of Oakland TAZ and TLN demand


,small_taz,medium_taz,large_taz,small_tln,medium_tln,large_tln,total
965,42,24,18,0,511,"1,015","1,609"
966,18,10,8,0,0,0,35
988,182,50,15,0,0,0,246
TOTAL,241,83,41,0,511,"1,015","1,891"
